## Como usar este notebook

1. Coloque os arquivos dbf na pasta:
   dados_brutos/

2. Execute o notebook do início ao fim.

3. Os dados tratados serão gerados na pasta:
   dados_processados/


**1. Instalação e imports**

In [ ]:
!pip install dbfread unidecode pyarrow

import os
import glob
import gc
from pathlib import Path

import pandas as pd
from dbfread import DBF
from unidecode import unidecode


**2. Paths**


In [ ]:
# Diretório base = pasta onde está o notebook
BASE_DIR = Path(".").resolve()

# Pasta com os DBFs brutos
RAW_DIR = BASE_DIR / "dados_brutos"

# Pasta onde serão salvos os arquivos processados
PROC_DIR = BASE_DIR / "dados_processados"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Lista de arquivos DBF
dbf_files = [p for p in RAW_DIR.iterdir() if p.suffix.lower() == ".dbf"]
dbf_files

**3. Colunas do SINAN que serão mantidas**

In [ ]:
KEEP_COLS_BASE = [
 "DT_NOTIFIC",
 "NU_ANO",
 "SG_UF_NOT",
 "ID_MUNICIP",
 "ID_UNIDADE",
 "DT_OCOR",
 "NU_IDADE_N",
 "CS_SEXO",
 "CS_GESTANT",
 "CS_RACA",
 "CS_ESCOL_N",
 "SG_UF",
 "ID_MN_RESI",
 "ID_PAIS",
 "DT_INVEST",
 "ID_OCUPA_N",
 "SIT_CONJUG",
 "DEF_TRANS",
 "DEF_FISICA",
 "DEF_MENTAL",
 "DEF_VISUAL",
 "DEF_AUDITI",
 "TRAN_MENT",
 "TRAN_COMP",
 "DEF_OUT",
 "DEF_ESPEC",
 "SG_UF_OCOR",
 "ID_MN_OCOR",
 "HORA_OCOR",
 "LOCAL_OCOR",
 "LOCAL_ESPE",
 "OUT_VEZES",
 "LES_AUTOP",
 "VIOL_FISIC",
 "VIOL_PSICO",
 "VIOL_TORT",
 "VIOL_SEXU",
 "VIOL_TRAF",
 "VIOL_FINAN",
 "VIOL_NEGLI",
 "VIOL_INFAN",
 "VIOL_LEGAL",
 "VIOL_OUTR",
 "VIOL_ESPEC",
 "AG_FORCA",
 "AG_ENFOR",
 "AG_OBJETO",
 "AG_CORTE",
 "AG_QUENTE",
 "AG_ENVEN",
 "AG_FOGO",
 "AG_AMEACA",
 "AG_OUTROS",
 "AG_ESPEC",
 "SEX_ASSEDI",
 "SEX_ESTUPR",
 "CONS_ABORT",
 "CONS_GRAV",
 "CONS_DST",
 "CONS_SUIC",
 "CONS_MENT",
 "CONS_COMP",
 "CONS_ESTRE",
 "CONS_OUTR",
 "CONS_ESPEC",
 "LESAO_NAT",
 "LESAO_ESPE",
 "LESAO_CORP",
 "NUM_ENVOLV",
 "REL_SEXUAL",
 "REL_PAI",
 "REL_MAE",
 "REL_PAD",
 "REL_CONJ",
 "REL_EXCON",
 "REL_NAMO",
 "REL_EXNAM",
 "REL_FILHO",
 "REL_DESCO",
 "REL_IRMAO",
 "REL_CONHEC",
 "REL_CUIDA",
 "REL_PATRAO",
 "REL_INST",
 "REL_POL",
 "REL_PROPRI",
 "REL_OUTROS",
 "REL_ESPEC",
 "AUTOR_SEXO",
 "AUTOR_ALCO",
 "ENC_SAUDE",
 "ENC_TUTELA",
 "ENC_VARA",
 "ENC_ABRIGO",
 "ENC_SENTIN",
 "ENC_DEAM",
 "ENC_DPCA",
 "ENC_DELEG",
 "ENC_MPU",
 "ENC_MULHER",
 "ENC_CREAS",
 "ENC_IML",
 "ENC_OUTR",
 "ENC_ESPEC",
 "REL_TRAB",
 "REL_CAT",
 "CIRC_LESAO",
 "CLASSI_FIN",
 "EVOLUCAO",
 "DT_OBITO",
 "DT_DIGITA",
 "DT_TRANSUS",
 "DT_TRANSDM",
 "DT_TRANSSM",
 "DT_TRANSRM",
 "DT_TRANSRS",
 "DT_TRANSSE",
 "REL_MAD",
 "TPUNINOT",
 "ORIENT_SEX",
 "IDENT_GEN",
 "VIOL_MOTIV",
 "CICL_VID",
 "REDE_SAU",
 "ASSIST_SOC",
 "REDE_EDUCA",
 "ATEND_MULH",
 "CONS_TUTEL",
 "CONS_IDO",
 "DELEG_IDOS",
 "DIR_HUMAN",
 "MPU",
 "DELEG_CRIA",
 "DELEG_MULH",
 "DELEG",
 "INFAN_JUV",
 "DEFEN_PUBL",
 "DT_ENCERRA"
]

**4. Conversão de NU_IDADE_N → idade em anos**

In [ ]:
def idade_em_anos(cod):
    if pd.isna(cod):
        return None
    try:
        cod = int(cod)
    except:
        return None
    
    unidade = cod // 1000
    valor   = cod % 1000
    
    if unidade == 1:   # horas
        return valor / 8760
    if unidade == 2:   # dias
        return valor / 365
    if unidade == 3:   # meses
        return valor / 12
    if unidade == 4:   # anos
        return valor
    return None


**5. Função de processamento**

In [ ]:
def processar_dbf(path_dbf: Path):
    """
    Lê um DBF do SINAN-Violência e filtra:
      • UF = 29
      • município = Salvador (ID_MUNICIP = 292740)
      • sexo = F
      • NÃO violência autoprovocada (LES_AUTOP != '1')
    """
    nome = path_dbf.name

    # extrai ano do nome, ex.: VIOLBR14.dbf -> 2014
    try:
        ano_2d = int(nome[-6:-4])
        ano = 2000 + ano_2d
    except:
        print(f"[AVISO] Não consegui extrair ano de {nome}, pulando.")
        return None

    print(f"\n=== Processando {nome} (ano {ano}) ===")

    tabela = DBF(str(path_dbf), encoding="latin1", ignore_missing_memofile=True)

    registros = []
    total = mantidos = 0

    for rec in tabela:
        total += 1

        # --- SEXO ---
        sexo_raw = rec.get("CS_SEXO")
        sexo = str(sexo_raw).strip() if sexo_raw not in (None, "") else None
        if sexo != "F":
            continue

        # --- UF DE NOTIFICAÇÃO (SG_UF_NOT pode vir como 29, '29' ou 'BA') ---
        uf_raw = rec.get("SG_UF_NOT")
        uf_str = str(uf_raw).strip() if uf_raw not in (None, "") else ""

        uf_cod = None
        try:
            uf_cod = int(uf_str)
        except:
            pass

        # Aceita: código 29 OU sigla 'BA'
        if not (uf_cod == 29 or uf_str.upper() == "BA"):
            continue

        # --- MUNICÍPIO DE NOTIFICAÇÃO (ID_MUNICIP) ---
        mun_raw = rec.get("ID_MUNICIP")
        mun_str = str(mun_raw).strip() if mun_raw not in (None, "") else ""

        # Trata casos como '292740.0', '0292740', etc.
        if "." in mun_str:
            mun_str = mun_str.split(".")[0]

        # remove zeros à esquerda
        mun_str = mun_str.lstrip("0")

        # aceita qualquer coisa que comece com 292740 (IBGE de Salvador)
        if not mun_str.startswith("292740"):
            continue

        # --- LESÃO AUTOPROVOCADA (LES_AUTOP) ---
        les_raw = rec.get("LES_AUTOP")
        les_aut = str(les_raw).strip() if les_raw not in (None, "") else None

        # você quer TUDO que NÃO é autoprovocada
        if les_aut == "1":
            continue

        mantidos += 1
        novo = {}

        for col in KEEP_COLS_BASE:
            if col in tabela.field_names:
                novo[col] = rec.get(col, None)
            else:
                novo[col] = None

        novo["ano"] = ano
        novo["IDADE_ANOS"] = idade_em_anos(rec.get("NU_IDADE_N"))

        registros.append(novo)

    print(f"Total lidos: {total}")
    print(f"Registros mantidos: {mantidos}")

    if not registros:
        return None

    return pd.DataFrame(registros)

**6. Processar todos os anos e salvar em `dados_processados/`**


In [ ]:
for path_dbf in sorted(dbf_files):
    df_ano = processar_dbf(path_dbf)
    if df_ano is None or df_ano.empty:
        continue

    ano = int(df_ano["ano"].iloc[0])
    out_path = PROC_DIR / f"sinan_SSA_mulheres_{ano}.parquet"
    df_ano.to_parquet(out_path, index=False)
    print(f"Salvo: {out_path} -> {df_ano.shape}")

    del df_ano
    gc.collect()

**7. Juntar todos os anos tratados**


In [ ]:
parquet_files = sorted(PROC_DIR.glob("sinan_SSA_mulheres_*.parquet"))
parquet_files

dfs = [pd.read_parquet(f) for f in parquet_files]
sinan = pd.concat(dfs, ignore_index=True)
sinan.shape

sinan.head()

**8. Criar faixa etária**

In [ ]:
bins = [0, 9, 14, 19, 24, 34, 44, 59, 120]
labels = ["0-9", "10-14", "15-19", "20-24", "25-34", "35-44", "45-59", "60+"]

sinan["FAIXA_ETARIA"] = pd.cut(sinan["IDADE_ANOS"], bins=bins, labels=labels)
sinan["FAIXA_ETARIA"].value_counts()

**9. Recodificação das variáveis**


In [ ]:
def aplicar_mapa(df, col_cod, col_desc, mapa):
    """
    Cria uma coluna descritiva (col_desc) a partir de uma coluna codificada (col_cod)
    usando um dicionário de mapeamento (mapa).
    """
    if col_cod not in df.columns:
        print(f"[AVISO] Coluna {col_cod} não encontrada, ignorando.")
        return df
    
    df[col_cod] = df[col_cod].astype(str).str.strip()
    df[col_desc] = df[col_cod].map(mapa)
    return df


# 1) DADOS DEMOGRÁFICOS
map_cs_sexo = {
    "M": "Masculino",
    "F": "Feminino",
    "I": "Ignorado"
}

map_cs_raca = {
    "1": "Branca",
    "2": "Preta",
    "3": "Amarela",
    "4": "Parda",
    "5": "Indígena",
    "9": "Ignorado"
}

map_cs_escol = {
    "1":  "1ª a 4ª série incompleta do EF",
    "2":  "4ª série completa do EF",
    "3":  "5ª a 8ª série incompleta do EF",
    "4":  "Ensino fundamental completo",
    "5":  "Ensino médio incompleto",
    "6":  "Ensino médio completo",
    "7":  "Educação superior incompleta",
    "8":  "Educação superior completa",
    "9":  "Ignorado",
    "10": "Não se aplica"
}

map_cs_gestant = {
    "1": "1º trimestre",
    "2": "2º trimestre",
    "3": "3º trimestre",
    "4": "Idade gestacional ignorada",
    "5": "Não",
    "6": "Não se aplica",
    "9": "Ignorado"
}

map_sit_conjug = {
    "1": "Solteiro(a)",
    "2": "Casado(a)/União consensual",
    "3": "Viúvo(a)",
    "4": "Separado(a)",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_orient_sex = {
    "1": "Heterossexual",
    "2": "Homossexual (gay/lésbica)",
    "3": "Bissexual",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_ident_gen = {
    "1": "Travesti",
    "2": "Transexual Mulher",
    "3": "Transexual Homem",
    "8": "Não se aplica",
    "9": "Ignorado"
}


# 2) DEFICIÊNCIA / TRANSTORNO
map_def_sim_nao = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_def_sim_nao_8 = {
    "1": "Sim",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

# 3) LOCAL / ZONA / REPETIÇÃO / AUTOPROVOCADA
map_local_ocor = {
    "1":  "Residência",
    "01": "Residência",
    "2":  "Habitação coletiva",
    "02": "Habitação coletiva",
    "3":  "Escola",
    "03": "Escola",
    "4":  "Local de prática esportiva",
    "04": "Local de prática esportiva",
    "5":  "Bar ou similar",
    "05": "Bar ou similar",
    "6":  "Via pública",
    "06": "Via pública",
    "7":  "Comércio/Serviços",
    "07": "Comércio/Serviços",
    "8":  "Indústrias/Construção",
    "08": "Indústrias/Construção",
    "9":  "Outro",
    "09": "Outro",
    "99": "Ignorado"
}

map_zona_ocor = {
    "1": "Urbana",
    "2": "Rural",
    "3": "Periurbana",
    "9": "Ignorado"
}

map_out_vezes = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_les_autop = {
    "1": "Sim (autoprovocada)",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

# 4) MOTIVO DA VIOLÊNCIA
map_viol_motiv = {
    "1":  "Sexismo",
    "2":  "LGBTfobia (Homo/Lesbo/Bi/Transfobia)",
    "3":  "Racismo",
    "4":  "Intolerância religiosa",
    "5":  "Xenofobia",
    "6":  "Conflito geracional",
    "7":  "Situação de rua",
    "8":  "Deficiência",
    "9":  "Outros",
    "88": "Não se aplica",
    "99": "Ignorado"
}

# 5) TIPOS DE VIOLÊNCIA (já tinha algo parecido, mas deixo padronizado)
map_viol = {
    "1": "Sim",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

cols_viol = [
    "VIOL_FISIC", "VIOL_PSICO", "VIOL_TORT", "VIOL_SEXU",
    "VIOL_TRAF", "VIOL_FINAN", "VIOL_NEGLI", "VIOL_INFAN",
    "VIOL_LEGAL", "VIOL_OUTR", "VIOL_ESPEC"
]

# 6) MEIO DE AGRESSÃO
cols_meio_agressao = [
    "AG_FORCA", "AG_ENFOR", "AG_OBJETO", "AG_CORTE",
    "AG_QUENTE", "AG_ENVEN", "AG_FOGO", "AG_AMEACA",
    "AG_OUTROS"
]
# todos seguem 1=Sim, 2=Não, 9=Ignorado -> usa map_viol

# 7) ENCAMINHAMENTOS / REDE
cols_encaminhamento = [
    "ENC_SAUDE", "ASSIST_SOC", "REDE_EDUCA", "ATEND_MULH",
    "CONS_TUTEL", "CONS_IDO", "DELEG_IDOS", "DIR_HUMAN",
    "MPU", "DELEG_CRIA", "DELEG_MULH", "DELEG",
    "INFAN_JUV", "DEFEN_PUBL", "ENC_TUTELA", "ENC_VARA",
    "ENC_ABRIGO", "ENC_SENTIN", "ENC_DEAM", "ENC_DPCA",
    "ENC_DELEG", "ENC_MPU", "ENC_MULHER", "ENC_CREAS",
    "ENC_IML", "ENC_OUTR", "ENC_ESPEC", "REDE_SAU"
]

# 8) RELAÇÃO AUTOR–VÍTIMA
cols_relacao = [
    "REL_PAI", "REL_MAE", "REL_PAD", "REL_MAD",
    "REL_CONJ", "REL_EXCON", "REL_NAMO", "REL_EXNAM",
    "REL_FILHO", "REL_IRMAO", "REL_CONHEC", "REL_DESCO",
    "REL_CUIDA", "REL_PATRAO", "REL_INST", "REL_POL",
    "REL_PROPRI", "REL_OUTROS"
]

map_rel = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

# 9) VIOLÊNCIA RELACIONADA AO TRABALHO / CAT / Nº ENVOLVIDOS
map_rel_trab = {
    "1": "Sim",
    "2": "Não",
    "9": "Ignorado"
}

map_rel_cat = {
    "1": "Sim",
    "2": "Não",
    "8": "Não se aplica",
    "9": "Ignorado"
}

map_num_envolv = {
    "1": "Um",
    "2": "Dois ou mais",
    "9": "Ignorado"
}

# ------------------------------------------------------------
# APLICANDO OS MAPAS
# ------------------------------------------------------------

# Demográficas
sinan = aplicar_mapa(sinan, "CS_SEXO",    "CS_SEXO_DESC",    map_cs_sexo)
sinan = aplicar_mapa(sinan, "CS_RACA",    "CS_RACA_DESC",    map_cs_raca)
sinan = aplicar_mapa(sinan, "CS_ESCOL_N", "CS_ESCOL_DESC",   map_cs_escol)
sinan = aplicar_mapa(sinan, "CS_GESTANT", "CS_GESTANT_DESC", map_cs_gestant)
sinan = aplicar_mapa(sinan, "SIT_CONJUG", "SIT_CONJUG_DESC", map_sit_conjug)
sinan = aplicar_mapa(sinan, "ORIENT_SEX","ORIENT_SEX_DESC", map_orient_sex)
sinan = aplicar_mapa(sinan, "IDENT_GEN",  "IDENT_GEN_DESC",  map_ident_gen)

# Deficiência / transtorno
sinan = aplicar_mapa(sinan, "DEF_TRANS",  "DEF_TRANS_DESC",  map_def_sim_nao)
for col in ["DEF_FISICA", "DEF_MENTAL", "DEF_VISUAL", "DEF_AUDITI", "TRAN_MENT", "TRAN_COMP", "DEF_OUT", "DEF_ESPEC"]:
    sinan = aplicar_mapa(sinan, col, col + "_DESC", map_def_sim_nao_8)

# Local, zona, repetição, autoprovocada
sinan = aplicar_mapa(sinan, "LOCAL_OCOR", "LOCAL_OCOR_DESC", map_local_ocor)
sinan = aplicar_mapa(sinan, "ZONA_OCOR",  "ZONA_OCOR_DESC",  map_zona_ocor)  if "ZONA_OCOR" in sinan.columns else sinan
sinan = aplicar_mapa(sinan, "OUT_VEZES",  "OUT_VEZES_DESC",  map_out_vezes)
sinan = aplicar_mapa(sinan, "LES_AUTOP",  "LES_AUTOP_DESC",  map_les_autop)

# Motivo da violência
sinan = aplicar_mapa(sinan, "VIOL_MOTIV","VIOL_MOTIV_DESC", map_viol_motiv)

# Tipos de violência
for col in cols_viol:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_viol)

# Meio de agressão
for col in cols_meio_agressao:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_viol)

# Encaminhamentos / rede
for col in cols_encaminhamento:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_viol)  # 1/2/9

# Relação autor–vítima
for col in cols_relacao:
    if col in sinan.columns:
        sinan = aplicar_mapa(sinan, col, col + "_DESC", map_rel)

# Trabalho / CAT / nº envolvidos
sinan = aplicar_mapa(sinan, "REL_TRAB",  "REL_TRAB_DESC",  map_rel_trab)  if "REL_TRAB" in sinan.columns else sinan
sinan = aplicar_mapa(sinan, "REL_CAT",   "REL_CAT_DESC",   map_rel_cat)   if "REL_CAT"  in sinan.columns else sinan
sinan = aplicar_mapa(sinan, "NUM_ENVOLV","NUM_ENVOLV_DESC",map_num_envolv)if "NUM_ENVOLV" in sinan.columns else sinan

# ------------------------------------------------------------
# COLUNA CONSOLIDADA: RELAÇÃO COM O AUTOR (texto único)
# ------------------------------------------------------------

def consolidar_relacao(row):
    etiquetas = []
    mapa_nomes = {
        "REL_PAI_DESC":      "Pai",
        "REL_MAE_DESC":      "Mãe",
        "REL_PAD_DESC":      "Padrasto",
        "REL_MAD_DESC":      "Madrasta",
        "REL_CONJ_DESC":     "Cônjuge",
        "REL_EXCON_DESC":    "Ex-cônjuge",
        "REL_NAMO_DESC":     "Namorado(a)",
        "REL_EXNAM_DESC":    "Ex-namorado(a)",
        "REL_FILHO_DESC":    "Filho(a)",
        "REL_IRMAO_DESC":    "Irmão(ã)",
        "REL_CONHEC_DESC":   "Amigo/Conhecido",
        "REL_DESCO_DESC":    "Desconhecido",
        "REL_CUIDA_DESC":    "Cuidador(a)",
        "REL_PATRAO_DESC":   "Patrão/Chefe",
        "REL_INST_DESC":     "Instituição",
        "REL_POL_DESC":      "Policial/Agente do Estado",
        "REL_PROPRI_DESC":   "Proprietário",
        "REL_OUTROS_DESC":   "Outro"
    }
    for col_desc, nome in mapa_nomes.items():
        if col_desc in row.index and row[col_desc] == "Sim":
            etiquetas.append(nome)
    return ", ".join(etiquetas) if etiquetas else None

sinan["RELACAO_AUTOR"] = sinan.apply(consolidar_relacao, axis=1)


**10. Exportar dataset final**


In [ ]:
out_parquet = PROC_DIR / "sinan_final_SSA_mulheres.parquet"
out_csv     = PROC_DIR / "sinan_final_SSA_mulheres.csv"

sinan.to_parquet(out_parquet, index=False)
sinan.to_csv(out_csv, sep=";", index=False)

out_parquet, out_csv